<a href="https://colab.research.google.com/github/UmymaM/AQI-Predictor/blob/main/face_verification_w_Siamese_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [3]:
!kaggle datasets download -d atulanandjha/lfwpeople

Dataset URL: https://www.kaggle.com/datasets/atulanandjha/lfwpeople
License(s): GNU Lesser General Public License 3.0
100% 232M/232M [00:01<00:00, 157MB/s]



In [4]:
import zipfile
zipfile=zipfile.ZipFile('lfwpeople.zip') #unzipping the lfwpeople file
zipfile.extractall()
zipfile.close()

In [5]:
import tarfile

# Path to the tgz file
tgz_file_path = '/content/lfw-funneled.tgz'

# Directory to extract to
extract_dir = '/content/'

# Open the tgz file in read mode ('r:gz')
with tarfile.open(tgz_file_path, 'r:gz') as tar:
    # Extract all contents to the specified directory
    tar.extractall(path=extract_dir)

/tmp/ipykernel_16451/4013497105.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


In [ ]:
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), #no augmentation for test ds
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [7]:
import os

In [24]:
labeled_faces_dict={}
# filtering files
for file in os.listdir('/content/lfw_funneled'):
    file_path = os.path.join('/content/lfw_funneled', file)
    images=[]
    if os.path.isdir(file_path):
        # print(f"Length of dir {file}: {len(os.listdir(file_path))}")
        # continue
        for image in os.listdir(file_path):
          image=os.path.join(file_path,image)
          images.append(image)
        labeled_faces_dict[file]=images
    else:
        print(f"Skipping file: {file}")

Skipping file: pairs_01.txt
Skipping file: pairs_02.txt
Skipping file: pairs.txt
Skipping file: pairs_04.txt
Skipping file: pairs_07.txt
Skipping file: pairs_06.txt
Skipping file: pairs_05.txt
Skipping file: pairs_10.txt
Skipping file: pairs_03.txt
Skipping file: pairs_09.txt
Skipping file: pairs_08.txt


In [19]:
len(labeled_faces_dict.keys())

5749

In [25]:
labeled_faces_dict.get("Charles_Schumer")

['/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0002.jpg',
 '/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0001.jpg']